In [19]:
import pandas as pd
from datasets import Dataset
import torch

# Load dataset
df = pd.read_csv("news_data_1M_lines.txt", sep="\t", header=None, on_bad_lines="skip")
df.columns = ["incorrect", "correct"]
df = df.dropna().drop_duplicates()
df["incorrect"] = df["incorrect"].str.strip()
df["correct"] = df["correct"].str.strip()

print("Samples:", len(df))
display(df.head())

dataset = Dataset.from_pandas(df)

Samples: 1002046


,incorrect,correct
0,"प्रतोनध, नगपुर पन ठेे बद असतनह सध्य मोठ्य प्रम...","प्रतिनिधी, नागपूर पान ठेले बंद असतानाही सध्या ..."
1,"मत्र, ह खर्र तयर करण्यसठि बूरश चंढलेल्य सूपरिच...","मात्र, हा खर्रा तयार करण्यासाठी बुरशी चढलेल्या..."
2,बद असतन गटखह नगपरत मोठ्य प्रमणत यते असल्यचंेहि...,बंदी असताना गुटखाही नागपुरात मोठ्या प्रमाणात य...
3,अअन्न वा षऔध प्रशसन वीभगने केलाल्य तपसणत ७ु१ न...,अन्न व औषध प्रशासन विभागाने केलेल्या तपासणीत ७...
4,"म्य प्दरेश, छत्तिसगढ, आध्र प्रेदश य भगतुन मल ब...","मध्य प्रदेश, छत्तीसगढ, आंध्र प्रदेश या भागातून..."


In [ ]:
import re

def clean_text(text):
    # Remove unwanted sequences like NUUM, numbers mixed with letters
    text = re.sub(r"NU[0-9०-९]+", "", text)
    text = re.sub(r"[^\w\s.,।'’\"]+", "", text)  # keep only words, spaces, punctuation
    text = re.sub(r"\s+", " ", text)  # replace multiple spaces with single space
    return text.strip()

df["incorrect"] = df["incorrect"].apply(clean_text)
df["correct"] = df["correct"].apply(clean_text)

print(df.head(10))

In [13]:


dataset = Dataset.from_pandas(df)

# Train/Validation split
split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(train_dataset, val_dataset)

Dataset({
    features: ['incorrect', 'correct', '__index_level_0__'],
    num_rows: 54000
}) Dataset({
    features: ['incorrect', 'correct', '__index_level_0__'],
    num_rows: 6000
})


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/IndicBART")

def tokenize_function(example):
    # Encode the input (incorrect sentence)
    model_inputs = tokenizer(
        example["incorrect"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    # Encode the target (correct sentence)
    labels = tokenizer(
        example["correct"],
        max_length=64,
        padding="max_length",
        truncation=True
    )["input_ids"]

    # Replace PAD tokens with -100 so they are ignored in loss computation
    labels = [(token if token != tokenizer.pad_token_id else -100) for token in labels]

    model_inputs["labels"] = labels
    return model_inputs

# Map over datasets
tokenized_train = train_dataset.map(tokenize_function, batched=False)
tokenized_val = val_dataset.map(tokenize_function, batched=False)


Map:   0%|          | 0/54000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [17]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/IndicBART")
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,   
    gradient_accumulation_steps=4,  
    num_train_epochs=3,              
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=True,
    dataloader_num_workers=4
)

model.gradient_checkpointing_enable()  

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer
)

trainer.train()

model.save_pretrained("lang_analyzer_model")
tokenizer.save_pretrained("lang_analyzer_model")

C:\Users\samar\AppData\Local\Temp\ipykernel_19864\558103624.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.000000,nan


KeyboardInterrupt: 

In [ ]:
sample = val_dataset[0]
input_text = sample["incorrect"]
print("Input:", input_text)
print("Expected:", sample["correct"])

inputs = tokenizer(input_text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_length=64)
print("Prediction:", tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


tokenizer = AutoTokenizer.from_pretrained("lang_analyzer_model")
model = AutoModelForSeq2SeqLM.from_pretrained("lang_analyzer_model")

test_sentences = ["मल शाळत जाायचं आहे", "तो घरी नाही गेलं"]


inputs = tokenizer(test_sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
inputs.pop("token_type_ids", None) 


outputs = model.generate(**inputs, max_length=64, num_beams=5, early_stopping=True)


for sent, out in zip(test_sentences, outputs):
    pred = tokenizer.decode(out, skip_special_tokens=True)
    print("Incorrect:", sent)
    print("Corrected:", pred)
    print()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("lang_analyzer_model")
model = AutoModelForSeq2SeqLM.from_pretrained("lang_analyzer_model")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Tokenize inputs
tokenized = tokenizer(
    inputs,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)

# ✅ Remove token_type_ids (IndicBART doesn’t use it)
tokenized.pop("token_type_ids", None)

# Move tensors to GPU/CPU
tokenized = {k: v.to(device) for k, v in tokenized.items()}

# Generate predictions
outputs = model.generate(**tokenized, max_length=64, num_beams=5)
preds = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [ ]:
import evaluate
bleu = evaluate.load("bleu")
print("BLEU Score:", bleu.compute(predictions=preds,references=[[r] for r in refs])["bleu"])